# Week 1 Day 5 — LangChain LCEL + Gradio 챗봇 실습

**날짜:** 2026-03-14 (토, 주말 프로젝트)  
**주제:** Runnable (Parallel / Branch) + Gradio로 사내 FAQ 챗봇 만들기

---

## 오늘의 목표

1. **LCEL의 조력자 Runnable** 익히기 — `RunnableParallel`, `RunnableBranch`, `RunnablePassthrough`
2. **Gradio**로 파이썬 함수를 웹 UI로 띄우기 — `Interface`, `ChatInterface`, `Blocks`
3. **LLM + Gradio 결합** — 대화 히스토리 관리하는 챗봇
4. **FAQ 데이터 주입 챗봇** — 지식 베이스를 프롬프트에 넣어주는 가장 단순한 형태의 RAG

## 선생님이 쓰신 비유 (강의 중)

> "LCEL의 `|` 파이프는 물 흐르듯 앞 결과가 뒤로 넘어가는 거고,  
> Runnable은 그 물길을 **갈라주거나(Parallel)**, **분기시키거나(Branch)**, **그대로 통과시키는(Passthrough)** 수도꼭지다."

오늘은 그 수도꼭지들을 하나씩 열어보는 날.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w1_api_and_chain/llm_260314_langChain_practice2.ipynb)

## 0. Colab 환경 설정

Colab에서 실행하는 경우 아래 셀을 한 번 실행해 주세요.  
로컬에서 하면 `.env` 파일에 `OPENAI_API_KEY=...`를 넣고 `load_dotenv()`로 불러오면 됩니다.

> **비유:** API 키는 '출입증'과 같음. 실습 시작 전 목에 걸어두지 않으면 OpenAI 건물 안에 못 들어감.

In [ ]:
# Colab 환경 전용 설치 (로컬이라면 주석 처리)
!pip install -q langchain-openai langchain-core python-dotenv gradio

## 1. 핵심 클래스 import

- `ChatOpenAI` : OpenAI GPT와 대화하는 LangChain 래퍼 (전화기 역할)
- `ChatPromptTemplate` : 프롬프트 템플릿 (편지 양식)
- `StrOutputParser` : LLM의 응답에서 `.content` 문자열만 쏙 뽑아주는 파서 (알맹이 추출기)

In [ ]:
# 어제(3/13)에 이어서, 오늘도 쓸 공통 클래스들
from langchain_openai import ChatOpenAI              # LLM 연결용 클래스
from langchain_core.prompts import ChatPromptTemplate # 프롬프트 템플릿
from langchain_core.output_parsers import StrOutputParser  # LLM 응답을 문자열로 추출

## 2. API 키 로드

로컬/Colab 양쪽 모두 작동하도록 `try/except` 스타일로 섞어 둠.  
민아가 쓰던 방식(`google.colab.userdata`)도 그대로 유지.

In [ ]:
import os
from dotenv import load_dotenv

# 1) 로컬 .env 파일에서 읽기 시도
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

# 2) Colab이라면 userdata에서 가져오기
if api_key is None:
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENAI_API_KEY')
        os.environ['OPENAI_API_KEY'] = api_key  # LangChain이 환경변수에서도 참조 가능하도록
    except Exception:
        pass

# 키가 제대로 들어왔는지 확인 (앞 10자리만 출력)
print('API key loaded:', api_key[:10] + '...' if api_key else '❌ 없음')

## 3. LCEL 복습 — 파이프로 이어지는 체인

어제까지 배운 기본 공식:

```python
chain = prompt | llm | parser
```

- **LCEL** = **L**ang**C**hain **E**xpression **L**anguage
- `|` 연산자는 리눅스 파이프(`ls | grep`)와 정확히 같은 철학 — 왼쪽 출력이 오른쪽 입력이 됨
- 앞 단계 결과가 뒤 단계로 자동 전달되는 **데이터 흐름 선언**

오늘은 이 파이프라인을 더 풍부하게 만들어주는 **Runnable** 친구들을 만난다.

In [ ]:
# chain = prompt | llm | parser
# LCEL : LangChain Expression Language
# LCEL을 도와주는 Runnable 친구들을 오늘 만나볼 예정

## 4. Runnable 3형제

| 이름 | 역할 | 비유 |
|---|---|---|
| `RunnableSequence` | 순차 실행 (사실 `\|` 파이프가 이거임) | 컨베이어 벨트 |
| `RunnableParallel` | 여러 체인 **동시** 실행 → dict로 반환 | 한 번에 여러 주방장에게 같은 재료 주고 다른 요리 시키기 |
| `RunnablePassthrough` | 입력을 **가공 없이** 그대로 전달 | 전달만 하는 택배 기사 |
| `RunnableBranch` | 조건 따라 다른 체인으로 **분기** | 회사 전화 안내 "1번은 기술지원, 2번은 요금문의..." |

In [ ]:
# Runnable 3형제
# RunnableSequence  : 순차 실행 (| 파이프)
# RunnableParallel  : 병렬 실행 (여러 체인 동시에)
# RunnablePassthrough : 그대로 통과
# RunnableBranch    : 조건 분기

## 5. `RunnableParallel` — 병렬 체인

같은 입력 `{text}`를 받아 **요약**과 **키워드 추출**을 **동시에** 수행.  
결과는 `{'summary': ..., 'keywords': ...}` 형태의 dict로 나옴.

> **주의:** 아래 셀에서는 아직 LLM을 연결하지 않아서 `ChatPromptValue`(프롬프트 상태)만 찍힘.  
> 제대로 실행되는 모습을 보려면 `| llm | parser`를 붙여야 함.

In [ ]:
from langchain_core.runnables import RunnableParallel

# 같은 입력을 두 개의 다른 프롬프트로 동시에 처리
parallel_chain = RunnableParallel(
    summary  = ChatPromptTemplate.from_template('{text}를 한 줄로 요약해 주세요.'),
    keywords = ChatPromptTemplate.from_template('{text}에서 키워드를 3개만 뽑아주세요.'),
)

In [ ]:
# invoke로 실행 — {'text': ...} 형태로 입력
result = parallel_chain.invoke({'text': 'Langchain은 LLM 기반 application 개발 프레임워크입니다.'})
result

## 6. LLM 초기화

이제부터 실제로 OpenAI에게 말을 걸 친구를 만들어 둠.  
모델은 가성비 좋은 `gpt-4o-mini` 사용.

In [ ]:
# Colab의 경우 api_key를 명시적으로 넘겨줘야 안정적
llm = ChatOpenAI(model='gpt-4o-mini', openai_api_key=api_key)

# 잘 연결됐는지 가벼운 테스트
llm.invoke('안녕하세요').content

## 7. `RunnableBranch` — 조건 분기 체인

고객 문의가 들어오면 내용에 따라 **다른 팀**으로 넘기는 상황.

- '오류', '에러' → 기술지원팀 체인
- '가격', '요금' → 요금관리팀 체인
- 그 외 → 일반상담팀 체인

> **비유:** 회사 대표번호 ARS 같음. "기술지원 1번, 요금문의 2번..."  
> 해당 번호로 눌리는 순간 그 팀의 시나리오(=체인)가 재생됨.

강의 중 선생님이 처음엔 `llm`과 `parser`를 붙이는 걸 빼먹어서 결과가 안 나왔던 장면이 있었음.  
→ **체인은 반드시 `prompt | llm | parser`까지 완성해야 응답이 생성된다**는 교훈.

In [ ]:
# RunnableBranch로 3개 팀 체인을 라우팅
from langchain_core.runnables import RunnableBranch

parser = StrOutputParser()  # 문자열 추출기

# 팀별 체인 — 시스템 페르소나만 바꿔서 3벌 만듦
tech_chain    = ChatPromptTemplate.from_template('기술지원팀입니다 : {question}')     | llm | parser
billing_chain = ChatPromptTemplate.from_template('요금 관리 팀입니다 : {question}')   | llm | parser
general_chain = ChatPromptTemplate.from_template('일반 상담 팀입니다 : {question}')   | llm | parser

# 분기 조건 함수 — 지금은 키워드 하드코딩, 추후엔 LLM에게 맡길 수도
def route_logic(x):
    text = x['question']  # 딕셔너리에서 질문 꺼내기
    if ('오류' in text) or ('에러' in text):
        return 'technical'
    elif ('가격' in text) or ('요금' in text):
        return 'billing'
    return 'general'

# RunnableBranch는 (조건, 체인) 튜플 리스트 + 마지막 기본 체인 구조
branch = RunnableBranch(
    (lambda x: route_logic(x) == 'technical', tech_chain),
    (lambda x: route_logic(x) == 'billing',   billing_chain),
    general_chain,  # 마지막은 default
)

# 실제로 돌려보기
questions = ['프린터 오류가 났습니다', '월 요금이 얼마인가요?', '영업시간 알려주세요']
for q in questions:
    print(f"Q: {q}\nA: {branch.invoke({'question': q})}\n")

### 💡 분기 고도화 아이디어

지금은 `route_logic`을 하드코딩 키워드로 짰지만,  
현실에서는 "프린터 고장났어요", "출력이 안 돼요" 등 **키워드를 벗어난 표현**이 수두룩함.

→ 해결책: `route_logic` 자체를 LLM에게 맡기기!  
  LLM에게 "질문을 technical/billing/general 중 하나로 분류해줘" 라고 시키면 훨씬 유연해짐.

이건 3주차 '라우팅 체인'에서 본격적으로 다룰 예정.

---

# Part 2. Gradio — 챗봇 UI 만들기

## 8. Gradio란?

파이썬 함수 하나만 있으면 **웹 UI 앱**을 뚝딱 만들어주는 도구.

> **비유:** 도시락 가게에 비유하면,
> - **함수** = 레시피
> - **Gradio** = 레시피를 실제 도시락으로 포장해서 진열대에 올려주는 직원
> - **`share=True`** = 전단지 돌려서 가게 주소를 외부에 공개하는 옵션

핵심 컴포넌트 세 가지:
- `gr.Interface` : 입력 → 함수 → 출력 (가장 단순한 폼)
- `gr.ChatInterface` : 채팅용 특화 UI (히스토리 자동 관리)
- `gr.Blocks` : 레이아웃 자유도 최상 (Row/Column 조합)

In [ ]:
# 배포를 쉽게 도와주는 라이브러리 설치 (이미 설치돼 있으면 skip)
!pip install -q gradio

In [ ]:
import gradio as gr

## 9. `gr.Interface` — 가장 단순한 폼

함수 하나 + 입력/출력 컴포넌트 지정 → 끝.

In [ ]:
# 입력받은 이름에 인사말을 붙여주는 간단한 함수
def greet(name):
    return f'안녕하세요, {name}님'

demo1 = gr.Interface(
    fn     = greet,                          # 실행할 함수
    inputs = gr.Textbox(label='이름 입력'),   # 입력 컴포넌트
    outputs= gr.Textbox(label='인사말'),      # 출력 컴포넌트
    title  = '인사봇',
)

# share=True를 주면 외부 공개 URL이 생성됨 (Colab에선 자동으로 share=True 적용)
demo1.launch(share=True)

In [ ]:
# 실행 중인 Gradio 서버를 닫아주는 것 — 포트가 쌓이면 충돌 나기 때문
demo1.close()

## 10. `gr.ChatInterface` — 채팅 UI

함수 시그니처가 `(message, history)` 형태라면 자동으로 챗봇 UI가 붙음.

- `message` : 사용자가 방금 입력한 한 줄
- `history` : 지금까지의 대화 기록 (튜플 리스트 or 딕셔너리 리스트 — Gradio 버전별로 다름!)

`examples`를 넣어두면 대화창 하단에 추천 질문 버튼이 뜸 (카카오톡 생활톡 느낌).

In [ ]:
# 단순히 메시지를 echo만 하는 챗봇
def echo_bot(message, history):
    return f'Echo: {message}'

demo2 = gr.ChatInterface(
    fn       = echo_bot,
    title    = '에코챗봇',
    examples = ['안녕하세요', '오늘 날씨 어때요', 'FAQ 챗봇 테스트'],
)

demo2.launch(share=True)

In [ ]:
demo2.close()

## 11. `gr.Blocks` — 레이아웃 자유도 최상

버튼/행/열을 직접 배치하고 싶을 때 사용.  
`Row()` 안에 `Column()`을 넣어서 **격자 레이아웃**을 만든다.

> **비유:** 블록은 레고. `with gr.Row(): with gr.Column(): ...` 로 차곡차곡 쌓는 느낌.

- `scale` 파라미터로 가로 비율 조절 (2:1 식)
- `submit_btn.click(fn=..., inputs=..., outputs=...)` 로 버튼 동작 바인딩

In [ ]:
with gr.Blocks(title='custom layout') as demo3:
    gr.Markdown('custom layout demo')  # 상단 설명 영역
    with gr.Row():                      # 가로로 두 개 컬럼 배치
        with gr.Column(scale=2):        # 왼쪽 컬럼 (너비 2배)
            input_text = gr.Textbox(label='질문')
            submit_btn = gr.Button('전송')
        with gr.Column(scale=1):        # 오른쪽 컬럼 (너비 1배)
            category_output = gr.Textbox(label='카테고리')

    output_text = gr.Textbox(label='답변')

    # 버튼 클릭 시 실행될 함수 (아직은 LLM 없이 간단한 키워드 분기)
    def process(text):
        cat = '기술' if any(kw in text for kw in ['오류', '설치', '연결']) else '일반'
        return cat, f'[{cat}] {text}'

    # 버튼 이벤트 바인딩
    submit_btn.click(
        fn      = process,
        inputs  = input_text,
        outputs = [category_output, output_text],
    )

demo3.launch(share=True)

In [ ]:
demo3.close()

## 12. 히스토리 형식 — 튜플 vs 딕셔너리

Gradio의 `ChatInterface`는 **버전마다 history 형식이 다름**. 수업 중 한참 헤맸던 포인트!

| 형식 | 예시 | 해당 Gradio |
|---|---|---|
| 튜플 | `[('사용자', 'AI'), ...]` | 구버전 |
| 딕셔너리 | `[{'role': 'user', 'content': '...'}, ...]` | 신버전 (OpenAI 스타일) |

실제 챗봇을 만들 때는 이 형식에 맞춰 `for` 루프를 돌려야 함.

In [ ]:
# 튜플 형식 히스토리 순회 예시
history = [('a', 1), ('b', 2), ('c', 3)]
for human, ai in history:
    print(human, ai)

In [ ]:
# 딕셔너리 형식 히스토리 순회 예시 (신버전 Gradio / OpenAI API 스타일)
history = [
    {'role': 'user',      'content': 'abcde'},
    {'role': 'assistant', 'content': 'abcdef'},
    {'role': 'user',      'content': 'abcdefg'},
]
for msg in history:
    print(msg['role'], msg['content'])

## 13. LLM + ChatInterface 결합

드디어 **진짜 챗봇**. 요점:

1. `SystemMessage` → 챗봇 성격 부여
2. `history` → `HumanMessage`/`AIMessage` 리스트로 변환 (대화 맥락 유지)
3. 사용자의 새 메시지는 맨 마지막에 `HumanMessage`로 추가
4. `llm.invoke(messages).content` → 답변 문자열 반환

> **비유:** 앞쪽 대사를 다 적은 대본을 LLM에게 넘기면서 "다음 대사 이어서 써줘"라고 부탁하는 것.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

def chat_with_llm(message, history):
    # 1) 시스템 메시지 — 챗봇 페르소나 지정
    messages = [SystemMessage(content='당신은 친절한 한국어 어시스턴트입니다.')]

    # 2) 기존 히스토리를 LangChain 메시지 객체로 변환
    #    Gradio 버전에 따라 history 구조가 달라지므로 둘 다 대응
    for item in history:
        if isinstance(item, dict):  # 신버전: {'role': ..., 'content': ...}
            if item['role'] == 'user':
                messages.append(HumanMessage(content=item['content']))
            else:
                messages.append(AIMessage(content=item['content']))
        else:                        # 구버전: ('사용자', 'AI') 튜플
            human, ai = item
            messages.append(HumanMessage(content=human))
            messages.append(AIMessage(content=ai))

    # 3) 이번 턴 사용자 메시지 추가
    messages.append(HumanMessage(content=message))

    # 4) LLM에게 전체 메시지 리스트를 넘기고 .content만 반환
    return llm.invoke(messages).content

demo4 = gr.ChatInterface(
    fn       = chat_with_llm,
    title    = 'Chatbot',
    examples = ['안녕하세요', 'Python에 대해 알려주세요', '오늘 기분이 좋아요'],
)

demo4.launch(share=True)

In [ ]:
demo4.close()

---

# Part 3. FAQ 데이터 주입 챗봇 (원시적 RAG)

## 14. 지식 주입이란?

ChatGPT는 똑똑하지만 **우리 회사 내부 규정**은 모름.  
→ **프롬프트에 FAQ 데이터를 통째로 넣어서** '이 자료 참고해서 답해'라고 시킴.

> **비유:** 신입사원에게 사내 매뉴얼을 책상 위에 두고 "답변할 때 이거 보고 해"라고 하는 것과 같음.

**한계:** 데이터가 커지면 프롬프트 토큰이 폭발 → 다음 주(2주차)에 **임베딩 + 벡터스토어(FAISS)**로 업그레이드할 예정.

In [ ]:
# 사내 IT FAQ 데이터 — 원래는 DB/문서에서 불러오지만 오늘은 하드코딩
faq_data = [
    {'category': '계정', 'question': '비밀번호를 잊어버렸습니다. 어떻게 초기화하나요?',
     'answer':   "IT 포털(it.company.com)에서 '비밀번호 재설정' 버튼을 클릭하세요. 등록된 이메일로 재설정 링크가 발송됩니다."},
    {'category': '계정', 'question': '계정이 잠겼습니다. 어떻게 해제하나요?',
     'answer':   '5회 이상 비밀번호를 틀리면 계정이 잠깁니다. IT 헬프데스크(내선 1234)에 연락하세요.'},
    {'category': '계정', 'question': '신규 계정은 어떻게 만드나요?',
     'answer':   '신규 입사자는 인사팀에서 IT팀에 요청합니다. 입사 당일 계정 정보가 이메일로 발송됩니다.'},
    {'category': '계정', 'question': '2단계 인증(MFA)을 설정하려면?',
     'answer':   'IT 포털 > 보안 설정 > MFA 활성화에서 설정합니다. Google Authenticator 앱을 사용하세요.'},
]

# LLM 재확인 (이미 위에서 만들었지만 셀 독립 실행 대비)
llm = ChatOpenAI(model='gpt-4o-mini', openai_api_key=api_key)

In [ ]:
# 리스트를 Q/A 형태의 긴 문자열로 합쳐 프롬프트에 주입할 수 있게 만들기
faq_context = '\n'.join([f"Q: {item['question']}\nA: {item['answer']}" for item in faq_data])
faq_context

## 15. FAQ 챗봇 완성

프롬프트 전략:

- **시스템 메시지에 FAQ 데이터를 통째로 삽입**
- "데이터에 없는 내용은 헬프데스크로 안내하라"는 **안전장치**도 추가 (= hallucination 방지)

강의 중 선생님이 이걸 두고 하신 말:  
> "이게 RAG의 가장 원시적 형태. 다음 주엔 FAQ를 임베딩해서 관련 Q만 뽑아 넣는 '제대로 된 RAG'로 업그레이드한다."

In [ ]:
# 시스템 프롬프트에 FAQ를 박아넣은 템플릿
prompt = ChatPromptTemplate.from_messages([
    ('system',
     f"너는 사내 IT 지원팀 챗봇이야. 아래 제공된 FAQ 데이터를 바탕으로 사용자 질문에 친절하게 답해줘. "
     f"데이터에 없는 내용은 'IT 헬프데스크(1234번)으로 문의해주세요' 라고 안내해줘\n\n"
     f"[FAQ 데이터]\n{faq_context}"),
    ('human', '{question}'),
])

# 체인 구성 — prompt → llm → 문자열 추출
chain = prompt | llm | StrOutputParser()

# Gradio ChatInterface에 붙일 콜백 함수
def chat_response(message, history):
    # history는 이번 MVP에선 안 쓰고 매 질문을 독립적으로 처리 (stateless)
    return chain.invoke({'question': message})

demo = gr.ChatInterface(
    fn          = chat_response,
    title       = '사내지원챗봇',
    examples    = ['비밀번호 어떻게 초기화하나요', 'Wi-fi가 너무 느려요', '2단계 인증 어떻게 설정하나요'],
    description = '무엇을 도와드릴까요?',
)

demo.launch(share=True)

In [ ]:
# 실습 종료 후엔 꼭 서버를 닫아주기 (포트 정리)
# demo.close()

---

## ✅ 오늘의 정리

### 오늘 배운 것
1. **Runnable 3형제** — `Parallel`(병렬), `Branch`(분기), `Passthrough`(통과)
2. **Gradio** — `Interface`, `ChatInterface`, `Blocks` 3가지 빌딩 블록
3. **LLM 챗봇** — 히스토리를 LangChain 메시지로 변환해 맥락 유지
4. **FAQ 주입 챗봇** — 프롬프트에 지식을 통째로 박아넣는 원시적 RAG

### 이번 주(W1) 총정리
- **화(3/11)**: Python 기초 복습
- **수(3/12)**: OpenAI API 직접 호출 + 프롬프트 엔지니어링
- **목(3/13)**: LangChain LCEL 기초 (`prompt | llm | parser`)
- **금/토(3/14)**: Runnable 고급 + Gradio + FAQ 챗봇 ← **오늘**

### 2주차 예고
- **임베딩(Embedding)** — 문장을 벡터로 바꾸기
- **FAISS 벡터 스토어** — 수천 개 FAQ에서 유사한 것만 빠르게 찾기
- **제대로 된 RAG 파이프라인** — 오늘의 "FAQ 통째로 박기" 방식을 고도화

### 주말 프로젝트
> FAQ 데이터를 **주택청약 관련 Q&A**로 바꿔서 동일 구조의 챗봇 제작!

### Mina의 체크포인트
- [x] Runnable 3형제 각각 한 번씩 invoke 해보기
- [x] ChatInterface에서 history 튜플/dict 둘 다 처리되게 만들기
- [x] FAQ 챗봇 Gradio share URL 받아보기
- [ ] 주택청약 데이터로 FAQ 교체 → p1_weekend1 과제로 이어감